In [1]:
import pandas as pd
import os

In [2]:
class CEJCMetaInfo:
    def __init__(self, filename):
        self.filename = filename
        self.df = pd.read_csv(filename, sep=',', index_col=None)
    
    def get_session_id_list(self):
        return self.df['会話ID'].unique().tolist()
    
    def get_speaker_id_list(self):
        return self.df['話者ID改'].unique().tolist()
    
    def get_speaker_info_in_session(self, session_id):
        subdf = self.df[self.df['会話ID'] == session_id]
        if len(subdf) == 0:
            raise ValueError('No such session id: {}'.format(session_id))
        label2id = {}
        id2label = {}
        id2wavfilename = {}
        for i, row in subdf.iterrows():
            label2id[row['話者ラベル']] = row['話者ID改']
            id2label[row['話者ID改']] = row['話者ラベル']
            id2wavfilename[row['話者ID改']] = row['音声ファイル名']
        return {
            'label2id': label2id,
            'id2wavfilename': id2wavfilename
        }
    
        


In [3]:
cejc_meta_info = CEJCMetaInfo('./session_speaker_wav_table.csv')

In [5]:
session_id_list = cejc_meta_info.get_session_id_list()
cejc_meta_info.get_speaker_info_in_session(session_id_list[0])

{'label2id': {'IC01_玲子': 'C001_000',
  'IC02_夏樹': 'C001_001',
  'IC04_美沙': 'C001_002',
  'IC03_可奈子': 'C001_003',
  'IC05_美香': 'C001_004',
  'Z10A_店員': 'C001_010'},
 'id2wavfilename': {'C001_000': 'C001_001_IC01.wav',
  'C001_001': 'C001_001_IC02.wav',
  'C001_002': 'C001_001_IC04.wav',
  'C001_003': 'C001_001_IC03.wav',
  'C001_004': 'C001_001_IC05.wav',
  'C001_010': '--'}}

In [6]:
session_id_list

['C001_001',
 'C001_002',
 'C001_003',
 'C001_004',
 'C001_005',
 'C001_006',
 'C001_007',
 'C001_012',
 'C001_013',
 'C002_003',
 'C002_004',
 'C002_005',
 'C002_006a',
 'C002_006b',
 'C002_006c',
 'C002_007',
 'C002_008',
 'C002_013a',
 'C002_013b',
 'C002_014a',
 'C002_014b',
 'C002_015',
 'C002_016',
 'K001_003a',
 'K001_003b',
 'K001_004',
 'K001_008',
 'K001_009',
 'K001_010',
 'K001_011',
 'K001_013',
 'K001_014',
 'K001_016',
 'K001_017',
 'K001_019',
 'K002_003a',
 'K002_003b',
 'K002_004',
 'K002_007',
 'K002_010',
 'K002_012',
 'K002_014',
 'K002_015',
 'K002_016',
 'K002_017',
 'K002_018',
 'K003_002a',
 'K003_002b',
 'K003_002c',
 'K003_002d',
 'K003_003',
 'K003_005',
 'K003_006',
 'K003_008',
 'K003_012a',
 'K003_012b',
 'K003_013a',
 'K003_013b',
 'K003_014',
 'K003_017',
 'K004_001',
 'K004_005',
 'K004_007',
 'K004_008',
 'K004_010',
 'K004_011',
 'K004_012',
 'K004_013',
 'K004_015',
 'K004_017',
 'K004_020',
 'K004_022',
 'K005_001',
 'K005_002',
 'K005_003',
 'K005

In [7]:
CEJC_DIR = "/autofs/diamond2/share/corpus/CEJC"
CEJC2304_DIR = "/autofs/diamond2/share/corpus/CEJC2304"

In [13]:
# SOURCE_FILE_PATH = os.path.join(CEJC_DIR, "data/T014/T014_005/T014_005a-morphLUW.csv")
# SOURCE_FILE_PATH = os.path.join(CEJC_DIR, "data/C001/C001_003/C001_003-morphSUW.csv")
SOURCE_FILE_PATH = os.path.join(CEJC_DIR, "data/K002/K002_007/K002_007-morphSUW.csv")

In [14]:
datafarme = pd.read_csv(SOURCE_FILE_PATH, encoding="sjis")

In [15]:
datafarme.columns

Index(['会話ID', '短単位連番', '文頭フラグ', '話者ラベル', '書字形', '語彙素読み', '語彙素', '品詞', '活用型',
       '活用形', '語種', '語彙素細分類', '語形', 'タグ付き書字形', '発音形出現形', '発音', '発話単位の開始時刻',
       '発話単位の終了時刻', '転記単位の開始時刻', '転記単位の終了時刻', '仮名'],
      dtype='object')

In [84]:
# datafarme

In [85]:
def row2data(row) -> dict:
    """SUW（形態素短単位）形式の行データを辞書に変換する
    """
    return {
        "session_id": row["会話ID"],
        "sentence_start_flag": row["文頭フラグ"],
        "speaker_label": row["話者ラベル"],
        "pos": [row["品詞"]],
        "text": [row["書字形"]],
        "tagged_text": [row["タグ付き書字形"]],
        "pron_text": [row["発音形出現形"]],
        "pron": [row["発音"]],
        "time_start": float(row["発話単位の開始時刻"]),
        "time_end": float(row["発話単位の終了時刻"]),
        "alias_flag": [bool(row["仮名"])],
    }


In [86]:
def union_data(data1: dict, data2: dict) -> dict:
    """二つの発話データを統合する

    Args:
        data1 (dict): 発話データ1
        data2 (dict): 発話データ2

    Returns:
        dict: 統合された発話データ
    """
    assert data1["session_id"] == data2["session_id"], "session_id must be same, but {} and {}".format(data1["session_id"], data2["session_id"])
    assert data1["speaker_label"] == data2["speaker_label"], "speaker_label must be same, but {} and {}".format(data1["speaker_label"], data2["speaker_label"])

    return {
        "session_id": data1["session_id"],
        "sentence_start_flag": data1["sentence_start_flag"],
        "speaker_label": data1["speaker_label"],
        "pos": data1["pos"] + data2["pos"],
        "text": data1["text"] + data2["text"],
        "tagged_text": data1["tagged_text"] + data2["tagged_text"],
        "pron_text": data1["pron_text"] + data2["pron_text"],
        "pron": data1["pron"] + data2["pron"],
        "time_start": min(data1["time_start"], data2["time_start"]),
        "time_end": max(data1["time_end"], data2["time_end"]),
        "alias_flag": data1["alias_flag"] + data2["alias_flag"],
    }


In [87]:
th_g = 0.3   # ギャップの閾値．これを超えない場合は，同一発話とみなす
th_u = 10.0  # 初悪感の最大値，これを超える場合は別の発話とみなす

In [88]:
result = []
current_data = None
for i, row in datafarme.iterrows():
    data = row2data(row)
    if current_data is None:
        current_data = data
    else:
        if current_data["speaker_label"] != data["speaker_label"]:
            result.append(current_data)
            current_data = data
        elif data["sentence_start_flag"] == "I":
            current_data = union_data(current_data, data)
        elif data["time_start"] - current_data["time_end"] < th_g and data["time_end"] - current_data["time_start"] < th_u:
            current_data = union_data(current_data, data)
        else:
            result.append(current_data)
            current_data = data


In [98]:

def print_data_in_one_line(data: dict):
    out_text = ''
    for pos, tagged_text, pron, alias_flag in zip(data["pos"], data["tagged_text"], data["pron"], data["alias_flag"]):
        out = pron
        if alias_flag:
            out = f"<masked>({out})"
        if pos == "感動詞-フィラー":
            out = f"<F>{out}</F>"
        elif pos == "言いなおし":
            out = f"<D>{out}</D>"
        out_text += out

    print(f"{data['speaker_label']} {data['time_start']:.3f} {data['time_end']:.3f} {data['time_end']-data['time_start']:.3f} {''.join(data['text'])} {''.join(data['pron'])} {out_text}")

In [99]:
for data in result:
    print_data_in_one_line(data)

IC01_杉田 0.705 4.386 3.681 えーっと今週のさーちゃんの予定を教えてください エーットーコンシューノサーチャンノヨテーオオシエテクダサーイ <F>エーットー</F>コンシューノ<masked>(サー)チャンノヨテーオオシエテクダサーイ
IC02_沙織 4.649 5.579 0.930 今週 コンシュー コンシュー
IC01_杉田 8.802 13.429 4.627 あとえーとシーズンのあのなんだっけあれをもらってきてくれる アトエートシーズンノアノーナンダッケヤレオモラッテキテクレル アト<F>エート</F><masked>(シーズン)ノ<F>アノー</F>ナンダッケヤレオモラッテキテクレル
IC02_沙織 9.405 11.550 2.145 あ待って書くの忘れてたうん アマッテカクノワスレテタウン アマッテカクノワスレテタウン
IC01_杉田 13.429 13.722 0.293 また マタ マタ
IC02_沙織 13.812 14.310 0.498 うん ウン ウン
IC01_杉田 13.820 14.642 0.822 スケジュール帳 スケジュールチョー スケジュールチョー
IC02_沙織 14.633 15.511 0.878 忘れてた ワスレテタ ワスレテタ
IC01_杉田 14.967 18.661 3.694 できのうお話しすることができたのきのうイソ行ってないか デキノーオハナシスルコトガデキタノキノーイソイッテナイカ デキノーオハナシスルコトガデキタノキノーイソイッテナイカ
IC02_沙織 19.187 19.556 0.369 え エ エ
IC01_杉田 19.794 20.781 0.987 きのうは行ってないもんね キノーワイッテナイモンネ キノーワイッテナイモンネ
IC02_沙織 20.904 21.305 0.401 うん ウン ウン
IC01_杉田 22.877 26.144 3.267 で八月はもうお弁当取ろうかなと思ってんのね デハチガツワモーオベントートローカナトオモッテンノネ デハチガツワモーオベントートローカナトオモッテンノネ
IC02_沙織 26.584 27.003 0.419 うん ウン ウン
IC01_杉田 26.960 28.761 1.801 で一応八月いっぱいは行こうかなと デイチ

In [91]:
result[0]

{'session_id': 'K002_007',
 'sentence_start_flag': 'B',
 'speaker_label': 'IC01_杉田',
 'pos': ['感動詞-フィラー',
  '名詞-普通名詞-副詞可能',
  '助詞-格助詞',
  '名詞-固有名詞-人名-一般',
  '接尾辞-名詞的-一般',
  '助詞-格助詞',
  '名詞-普通名詞-サ変可能',
  '助詞-格助詞',
  '動詞-一般',
  '助詞-接続助詞',
  '動詞-非自立可能'],
 'text': ['えーっと', '今週', 'の', 'さー', 'ちゃん', 'の', '予定', 'を', '教え', 'て', 'ください'],
 'tagged_text': ['えーっと:',
  '今週',
  'の',
  '(R さー)',
  'ちゃん',
  'の',
  '予定',
  'を',
  '教え',
  'て',
  'くださ:い。'],
 'pron_text': ['エーット',
  'コンシュー',
  'ノ',
  'サー',
  'チャン',
  'ノ',
  'ヨテー',
  'オ',
  'オシエ',
  'テ',
  'クダサイ'],
 'pron': ['エーットー',
  'コンシュー',
  'ノ',
  'サー',
  'チャン',
  'ノ',
  'ヨテー',
  'オ',
  'オシエ',
  'テ',
  'クダサーイ'],
 'time_start': 0.705,
 'time_end': 4.386,
 'alias_flag': [False,
  False,
  False,
  True,
  False,
  False,
  False,
  False,
  False,
  False,
  False]}

In [92]:
utterances = []
current_data = {
    "session_id": None,
    "speaker_label": None,
    "start_time": None,
    "end_time": None,
    "tagged_text": None,
    "pron_app": None,
    "pron": None,
}
prev_start = -1
for i, row in datafarme.iterrows():
    if row["発話単位の開始時刻"] != prev_start and current_data["speaker_label"] != row["話者ラベル"]:
        if prev_start > 0:
            utterances.append(current_data)
        current_data = {
            "session_id": row["会話ID"],
            "speaker_label": row["話者ラベル"],
            "start_time": row["発話単位の開始時刻"],
            "end_time": row["発話単位の終了時刻"],
            "tagged_text": [row["タグ付き書字形"]],
            "pron_app": [row["発音形出現形"]],
            "pron": [row["発音"]],
        }
    else:
        current_data["tagged_text"].append(row["タグ付き書字形"])
        current_data["pron_app"].append(row["発音形出現形"])
        current_data["pron"].append(row["発音"])
    prev_start = row["発話単位の開始時刻"]


In [93]:
datafarme

,会話ID,短単位連番,文頭フラグ,話者ラベル,書字形,語彙素読み,語彙素,品詞,活用型,活用形,...,語彙素細分類,語形,タグ付き書字形,発音形出現形,発音,発話単位の開始時刻,発話単位の終了時刻,転記単位の開始時刻,転記単位の終了時刻,仮名
0,K002_007,1,B,IC01_杉田,えーっと,エート,えーと,感動詞-フィラー,NaN,NaN,...,NaN,エーット,えーっと:,エーット,エーットー,0.705,4.386,0.705,1.656,0
1,K002_007,2,I,IC01_杉田,今週,コンシュウ,今週,名詞-普通名詞-副詞可能,NaN,NaN,...,NaN,コンシュウ,今週,コンシュー,コンシュー,0.705,4.386,1.790,4.386,0
2,K002_007,3,I,IC01_杉田,の,ノ,の,助詞-格助詞,NaN,NaN,...,NaN,ノ,の,ノ,ノ,0.705,4.386,1.790,4.386,0
3,K002_007,4,I,IC01_杉田,さー,サア,サア,名詞-固有名詞-人名-一般,NaN,NaN,...,NaN,サア,(R さー),サー,サー,0.705,4.386,1.790,4.386,1
4,K002_007,5,I,IC01_杉田,ちゃん,チャン,ちゃん,接尾辞-名詞的-一般,NaN,NaN,...,NaN,チャン,ちゃん,チャン,チャン,0.705,4.386,1.790,4.386,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5271,K002_007,5272,I,IC01_杉田,ちゃん,チャン,ちゃん,接尾辞-名詞的-一般,NaN,NaN,...,NaN,チャン,ちゃん。,チャン,チャン,1751.868,1752.269,1751.868,1752.269,0
5272,K002_007,5273,B,IC02_沙織,え,エッ,えっ,感動詞-一般,NaN,NaN,...,NaN,エ,え。,エ,エ,1753.272,1753.459,1753.272,1753.459,0
5273,K002_007,5274,B,IC02_沙織,もう,モウ,もう,副詞,NaN,NaN,...,NaN,モウ,もう,モー,モー,1753.459,1754.453,1753.459,1754.453,0
5274,K002_007,5275,I,IC02_沙織,全然,ゼンゼン,全然,副詞,NaN,NaN,...,NaN,ゼンゼン,全然,ゼンゼン,ゼンゼン,1753.459,1754.453,1753.459,1754.453,0
